In [28]:
from ultralytics import YOLO

model = YOLO("/home/elicer/syh/best.pt")
'''
result = model.tune(
    data='/home/elicer/dataset/data_yolo.yaml',
    epochs=100,
    iterations=30,
    imgsz=1080,
    batch=4,
    space=search_space,
    optimizer="AdamW",
    resume=True
)
'''

'\nresult = model.tune(\n    data=\'/home/elicer/dataset/data_yolo.yaml\',\n    epochs=100,\n    iterations=30,\n    imgsz=1080,\n    batch=4,\n    space=search_space,\n    optimizer="AdamW",\n    resume=True\n)\n'

In [6]:
import os

os.rename('/home/elicer/dataset/labels_yolo', '/home/elicer/dataset/labels')

In [34]:
results = model.predict(source="/home/elicer/syh/[주차장]수지 신봉동 힐스테이트광교산아파트 주차장 출차영상(2023.9.2).mp4", save=True, save_txt=False, stream=False)



WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/5779) /home/elicer/syh/[주차장]수지 신봉동 힐스테이트광교산아파트 주차장 출차영상(2023.9.2).mp4: 640x1088 1 Disabled Icon, 6 Vehicles, 16.9ms
video 1/1 (frame 2/5779) /home/elicer/syh/[주차장]수지 신봉동 힐스테이트광교산아파트 주차장 출차영상(2023.9.2).mp4: 640x1088 1 Disabled Icon, 6 Vehicles, 13.2ms
video 1/1 (frame 3/5779) /home/elicer/syh/[주차장]수지 신봉동 힐스테이트광교산아파트 주차장 출차영상(2023.9.2).mp4: 640x1088 1 Disabled Icon, 6 Vehicles, 13.0ms
video 1/1 (frame 4/5779) /home/elicer/syh/[주차장]수지 신

RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "../c10/cuda/CUDACachingAllocator.cpp":830, please report a bug to PyTorch. 

In [16]:
import json
import os
import requests
import random

def filter_coco_annotations_only_single_class(original_json_path, target_class, output_json_path, max_images=500):
    with open(original_json_path, 'r') as f:
        coco = json.load(f)

    # 타겟 클래스 id
    catIds = [cat['id'] for cat in coco['categories'] if cat['name'] == target_class]
    if not catIds:
        raise ValueError(f"클래스 '{target_class}'가 COCO categories에 없습니다.")
    catId = catIds[0]

    # 이미지별 등장 클래스 집합 만들기
    imgid_to_cats = {}
    for ann in coco['annotations']:
        imgid = ann['image_id']
        if imgid not in imgid_to_cats:
            imgid_to_cats[imgid] = set()
        imgid_to_cats[imgid].add(ann['category_id'])

    # 해당 클래스만 있는 이미지 id만 추출
    selected_img_ids = [imgid for imgid, cats in imgid_to_cats.items() if cats == {catId}]

    # 500개로 제한 (랜덤 추출)
    if len(selected_img_ids) > max_images:
        random.seed(42)
        selected_img_ids = random.sample(selected_img_ids, max_images)

    # 해당 이미지에 속한 annotation만 필터링
    filtered_annotations = [ann for ann in coco['annotations'] if ann['image_id'] in selected_img_ids and ann['category_id'] == catId]
    filtered_images = [img for img in coco['images'] if img['id'] in selected_img_ids]
    filtered_categories = [cat for cat in coco['categories'] if cat['id'] == catId]

    filtered_coco = {
        "images": filtered_images,
        "annotations": filtered_annotations,
        "categories": filtered_categories,
        "info": coco.get('info', {}),
        "licenses": coco.get('licenses', [])
    }

    with open(output_json_path, 'w') as f:
        json.dump(filtered_coco, f)

    return output_json_path, filtered_images

def download_images(images, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    downloaded = 0
    for im in images:
        if 'coco_url' in im:
            img_url = im['coco_url']
            filename = im['file_name']
            filepath = os.path.join(save_dir, filename)
            if not os.path.exists(filepath):
                try:
                    img_data = requests.get(img_url, timeout=10).content
                    with open(filepath, 'wb') as handler:
                        handler.write(img_data)
                    downloaded += 1
                except Exception as e:
                    print(f"다운로드 실패: {img_url}, 오류: {e}")
    return downloaded

# ==== 실제 실행 ====

original_json = '/home/elicer/syh/annotations/instances_train2017.json'  # 실제 존재하는 경로로 수정
target_classes = ['motorcycle']

for cls in target_classes:
    output_json = f'/home/elicer/syh/annotations/filtered_{cls}_only.json'
    print(f"{cls}만 있는 이미지 필터링 시작...")
    filtered_json_path, filtered_images = filter_coco_annotations_only_single_class(
        original_json, cls, output_json, max_images=500
    )
    print(f"{cls}만 있는 이미지 개수(최대 500): {len(filtered_images)}")

    save_dir = f'/home/elicer/syh/images/{cls}_only'
    downloaded_count = download_images(filtered_images, save_dir)
    print(f"{cls}만 있는 이미지 다운로드 완료: {downloaded_count}개")


motorcycle만 있는 이미지 필터링 시작...
motorcycle만 있는 이미지 개수(최대 500): 280
motorcycle만 있는 이미지 다운로드 완료: 280개


In [ ]:
import os
import shutil

def copy_files_with_labels(label_dir, target_labels, dest_dir):
    if not os.path.exists(label_dir):
        print(f"라벨 폴더가 존재하지 않습니다: {label_dir}")
        return []
    os.makedirs(dest_dir, exist_ok=True)
    copied_files = []
    for filename in os.listdir(label_dir):
        if filename.endswith('.txt'):
            filepath = os.path.join(label_dir, filename)
            with open(filepath, 'r') as f:
                lines = f.readlines()
                labels = set()
                for line in lines:
                    if line.strip():
                        label = line.strip().split()[0]
                        labels.add(label)
                # target_labels 중 하나라도 포함되어 있으면 복사
                if any(label in target_labels for label in labels):
                    shutil.copy(filepath, os.path.join(dest_dir, filename))
                    copied_files.append(filename)
    print(f"복사된 파일 개수: {len(copied_files)}")
    return copied_files

# 예시 경로 및 라벨
label_dir = '/home/elicer/dataset/labels/train'  # 실제 라벨 폴더 경로로 수정
dest_dir = '/home/elicer/syh/label'  # 복사할 폴더
target_labels = {'0', '1', '3', '4'}

copied_files = copy_files_with_labels(label_dir, target_labels, dest_dir)


In [21]:
import json
import os

def coco_to_yolo_seg_from_filtered(filtered_coco, output_dir, class_mapping=None):
    cats = filtered_coco['categories']
    if class_mapping is None:
        class_mapping = {cat['id']: idx for idx, cat in enumerate(cats)}
    images = {img['id']: img for img in filtered_coco['images']}
    anns_per_image = {}
    for ann in filtered_coco['annotations']:
        if 'segmentation' not in ann or not ann['segmentation']:
            continue
        img_id = ann['image_id']
        if img_id not in anns_per_image:
            anns_per_image[img_id] = []
        anns_per_image[img_id].append(ann)
    os.makedirs(output_dir, exist_ok=True)
    for img_id, anns in anns_per_image.items():
        img_info = images[img_id]
        img_w = float(img_info['width'])
        img_h = float(img_info['height'])
        label_lines = []
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id not in class_mapping:
                continue
            cls_idx = class_mapping[cat_id]
            for seg in ann['segmentation']:
                norm_coords = []
                for i in range(0, len(seg), 2):
                    try:
                        x = float(seg[i]) / img_w
                        y = float(seg[i+1]) / img_h
                        norm_coords.extend([x, y])
                    except Exception:
                        continue
                if norm_coords:
                    line = f"{cls_idx} " + " ".join(f"{c:.6f}" for c in norm_coords)
                    label_lines.append(line)
        txt_filename = os.path.splitext(img_info['file_name'])[0] + '.txt'
        txt_path = os.path.join(output_dir, txt_filename)
        with open(txt_path, 'w') as f:
            f.write('\n'.join(label_lines))
    print(f"YOLO segmentation labels saved to {output_dir}")

# ===== 사용 예시 =====

original_json_path = '/home/elicer/syh/annotations/filtered_person_only.json'  # 실제로 존재하는 경로로 수정
output_dir = '/home/elicer/syh/data_only/lab_per'

# COCO category_id 1('person')을 YOLO 라벨 6으로 매핑
class_mapping = {1: 6}

if os.path.exists(original_json_path):
    with open(original_json_path, 'r') as f:
        filtered_coco = json.load(f)
    coco_to_yolo_seg_from_filtered(filtered_coco, output_dir, class_mapping=class_mapping)
else:
    print(f"파일이 존재하지 않습니다: {original_json_path}")


YOLO segmentation labels saved to /home/elicer/syh/data_only/lab_per


In [ ]:
import json
import os

def convert_coco_to_yolo_seg(coco_json_path, output_dir, class_mapping=None):
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)
    images = {img['id']: img for img in coco['images']}
    categories = {cat['id']: cat['name'] for cat in coco['categories']}
    os.makedirs(output_dir, exist_ok=True)
    if class_mapping is None:
        # COCO category_id → YOLO 인덱스 (순서대로)
        class_mapping = {cat_id: idx for idx, cat_id in enumerate(categories.keys())}
    annotations_per_image = {}
    for ann in coco['annotations']:
        if 'segmentation' not in ann or not ann['segmentation']:
            continue
        img_id = ann['image_id']
        if img_id not in annotations_per_image:
            annotations_per_image[img_id] = []
        annotations_per_image[img_id].append(ann)
    for img_id, anns in annotations_per_image.items():
        img_info = images[img_id]
        img_w = float(img_info['width'])
        img_h = float(img_info['height'])
        label_lines = []
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id not in class_mapping:
                continue
            cls_idx = class_mapping[cat_id]
            for seg in ann['segmentation']:
                norm_coords = []
                for i in range(0, len(seg), 2):
                    x = float(seg[i]) / img_w
                    y = float(seg[i+1]) / img_h
                    norm_coords.extend([x, y])
                line = f"{cls_idx} " + " ".join(f"{c:.6f}" for c in norm_coords)
                label_lines.append(line)
        txt_filename = os.path.splitext(img_info['file_name'])[0] + '.txt'
        txt_path = os.path.join(output_dir, txt_filename)
        with open(txt_path, 'w') as f:
            f.write('\n'.join(label_lines))
    print(f"YOLO segmentation labels saved to {output_dir}")

# ===== 사용 예시 =====
for i in range(0, len(seg), 2):
    try:
        x = float(seg[i]) / img_w
        y = float(seg[i+1]) / img_h
        norm_coords.extend([x, y])
    except ValueError:
        print(f"잘못된 좌표값: {seg[i]}, {seg[i+1]} (이미지: {img_info['file_name']})")
        norm_coords = []  # 잘못된 annotation은 무시
        break
if norm_coords:
    line = f"{cls_idx} " + " ".join(f"{c:.6f}" for c in norm_coords)
    label_lines.append(line)

coco_json_path = '/home/elicer/syh/annotations/filtered_instances.json'  # COCO 어노테이션 경로
output_dir = '/home/elicer/syh/data/labels/train'  # YOLO txt 저장 폴더
target_classes = ['person', 'car', 'motorcycle']

# 클래스 매핑: COCO category_id → YOLO 인덱스(0,1,2)
with open(coco_json_path) as f:
    cats = json.load(f)['categories']
cat_id_to_yolo_idx = {cat['id']: idx for idx, cat in enumerate(cats) if cat['name'] in target_classes}

convert_coco_to_yolo_seg(coco_json_path, output_dir, class_mapping=cat_id_to_yolo_idx)



ValueError: could not convert string to float: 'c'

In [ ]:
import os
import zipfile
import shutil
import json
import logging
import time
import psutil
import functools
# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("log_1.log", mode='a')
    ]
)
# 시간 측정 데코레이터
def time_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start
        logging.info(f"[TIME] {func.__name__} took {duration:.4f} seconds")
        return result
    return wrapper
# 메모리 사용량 측정 데코레이터
def memory_logger(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss / (1024 * 1024)
        result = func(*args, **kwargs)
        mem_after = process.memory_info().rss / (1024 * 1024)
        logging.info(f"[MEMORY] {func.__name__} used {mem_after - mem_before:.2f} MB")
        return result
    return wrapper
# === 함수 정의 ===
@time_logger
def extract_zips(root_dir, zip_suffix, callback, frq):
    try:
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in filenames:
                if fname.endswith(zip_suffix):
                    zip_path = os.path.join(dirpath, fname)
                    extract_dir = os.path.join(dirpath, fname[:-4])
                    os.makedirs(extract_dir, exist_ok=True)
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        zip_ref.extractall(extract_dir)
                    logging.info(f"Extracted: {zip_path} -> {extract_dir}")
                    callback(extract_dir, temp_image_dir if "camera.zip" in zip_suffix else temp_label_dir, frq)
    except Exception as e:
        logging.error(f"[extract_zips] Error: {e}", exc_info=True)
@time_logger
def clear_directory(root_dir):
    try:
        for filename in os.listdir(root_dir):
            file_path = os.path.join(root_dir, filename)
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        logging.info(f"Cleared directory: {root_dir}")
    except Exception as e:
        logging.error(f"[clear_directory] Error: {e}", exc_info=True)
@time_logger
def file_lists_equation(img_dir, label_dir):
    try:
        files1 = sorted([os.path.splitext(f)[0] for f in os.listdir(img_dir) if os.path.isfile(os.path.join(img_dir, f))])
        files2 = sorted([os.path.splitext(f)[0] for f in os.listdir(label_dir) if os.path.isfile(os.path.join(label_dir, f))])
        return files1 == files2
    except Exception as e:
        logging.error(f"[file_lists_equation] Error: {e}", exc_info=True)
        return False
@time_logger
def make_dirs(rst_dir):
    try:
        if os.path.exists(rst_dir):
            logging.info(f"Reusing existing directory: {rst_dir}")
        else:
            os.makedirs(rst_dir)
            logging.info(f"Created directory: {rst_dir}")
    except Exception as e:
        logging.error(f"[make_dirs] Error: {e}", exc_info=True)
@time_logger
def move_file(src, dst, files):
    try:
        src_path = os.path.join(src, files)
        dst_path = os.path.join(dst, files)
        shutil.move(src_path, dst_path)
        logging.info(f"Moved: {src_path} -> {dst_path}")
    except Exception as e:
        logging.error(f"[move_file] Error: {e} (src: {src_path}, dst: {dst_path})", exc_info=True)
@time_logger
def process_camera_files(extract_dir, temp_image_dir, frq):
    try:
        image_files = sorted(os.listdir(extract_dir))
        make_dirs(temp_image_dir)
        for i in range(0, len(image_files), frq):
            move_file(extract_dir, temp_image_dir, image_files[i])
        logging.info(f"Processed camera files from: {extract_dir}")
    except Exception as e:
        logging.error(f"[process_camera_files] Error: {e}", exc_info=True)
@time_logger
def process_segmentation_files(extract_dir, temp_label_dir, frq):
    try:
        make_dirs(temp_label_dir)
        for f in os.listdir(extract_dir):
            if f.startswith(('L', 'R')):
                os.remove(os.path.join(extract_dir, f))
        json_files = sorted(os.listdir(extract_dir))
        for i in range(0, len(json_files), frq):
            json_path = os.path.join(extract_dir, json_files[i])
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            w = data.get("meta", {}).get("size", {}).get("width", 1)
            h = data.get("meta", {}).get("size", {}).get("height", 1)
            image_name = os.path.splitext(os.path.basename(data["data_key"]))[0]
            label_path = os.path.join(temp_label_dir, image_name + ".txt")
            lines = []
            for obj in data["objects"]:
                class_id = class_to_id.get(obj["class_name"])
                for polygon in obj["annotation"]:
                    coords = []
                    for pt_list in polygon:
                        for pt in pt_list:
                            if isinstance(pt, dict) and "x" in pt and "y" in pt:
                                x = pt["x"] / w
                                y = pt["y"] / h
                                coords.extend([x, y])
                    if coords:
                        line = f"{class_id} " + " ".join([f"{c:.6f}" for c in coords])
                        lines.append(line)
            with open(label_path, "w") as f:
                f.write("\n".join(lines))
        logging.info(f"Processed segmentation files from: {extract_dir}")
    except Exception as e:
        logging.error(f"[process_segmentation_files] Error: {e}", exc_info=True)
@time_logger
def main():
    try:
        extract_zips(
            root_dir=root_dir,
            zip_suffix="camera.zip",
            callback=process_camera_files,
            frq=frq
        )
        extract_zips(
            root_dir=root_dir,
            zip_suffix="segmentation.zip",
            callback=process_segmentation_files,
            frq=frq
        )
        if file_lists_equation(temp_image_dir, temp_label_dir):
            image_files = os.listdir(temp_image_dir)
            label_files = os.listdir(temp_label_dir)
            for i in range(len(image_files)):
                move_file(temp_image_dir, output_image, image_files[i])
                move_file(temp_label_dir, output_label, label_files[i])
            logging.info("All files moved successfully.")
        else:
            logging.warning("File list mismatch between images and labels.")
        logging.info(f"Final match check: {file_lists_equation(output_image, output_label)}")
    except Exception as e:
        logging.critical(f"[main] Critical error: {e}", exc_info=True)
# === 전역 변수 정의 ===
class_names = [
    'Undefined Stuff', 'Wall', 'Driving Area', 'Non Driving Area', 'Parking Line',
    'Parking Area', 'No Parking Area', 'Big Notice', 'Pillar', 'Parking Area Number',
    'Disabled Icon', 'Women Icon', 'Compact Car Icon', 'Speed Bump', 'Parking Block',
    'Billboard', 'Toll Bar', 'Sign', 'No Parking Sign', 'Traffic Cone',
    'Fire Extinguisher', 'Undefined Object', 'Two-wheeled Vehicle', 'Vehicle',
    'Wheelchair', 'Stroller', 'Shopping Cart', 'Animal', 'Human'
]
class_to_id = {name: i for i, name in enumerate(class_names)}
root_dir = "/home/elicer/data_download" # 사용자 지정 디렉토리
output_image = "/home/elicer/datasets/images/train" # 사용자 지정 디렉토리
output_label = "/home/elicer/datasets/labels/train" # 사용자 지정 디렉토리
temp_image_dir = "/home/elicer/data_download/temp_images" # 자동 생성 디렉토리
temp_label_dir = "/home/elicer/data_download/temp_labels" # 자동 생성 디렉토리
frq = 4 # 사용자 지정 주기

if __name__ == "__main__":
    main()

2025-05-15 10:34:09,674 [INFO] Extracted: /home/elicer/data_download/181.실내_자율주차용_데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_객체인식(2Hz)_기타(주차타워)_20220901_14-18_02.camera.zip -> /home/elicer/data_download/181.실내_자율주차용_데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_객체인식(2Hz)_기타(주차타워)_20220901_14-18_02.camera
2025-05-15 10:34:09,690 [INFO] Created directory: /home/elicer/data_download/temp_images
2025-05-15 10:34:09,691 [INFO] [TIME] make_dirs took 0.0011 seconds
2025-05-15 10:34:09,693 [INFO] Moved: /home/elicer/data_download/181.실내_자율주차용_데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_객체인식(2Hz)_기타(주차타워)_20220901_14-18_02.camera/20220901_165216_18.png -> /home/elicer/data_download/temp_images/20220901_165216_18.png
2025-05-15 10:34:09,694 [INFO] [TIME] move_file took 0.0015 seconds
2025-05-15 10:34:09,695 [INFO] Moved: /home/elicer/data_download/181.실내_자율주차용_데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_객체인식(2Hz)_기타(주차타워)_20220901_14-18_02.camera/20220901_165218_48.png -> /home/elicer/data_download/temp_images/20220901_

In [5]:
def clear_directory(root_dir):
    for filename in os.listdir(root_dir):
        file_path = os.path.join(root_dir, filename)
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.remove(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)

clear_directory(root_dir)

In [1]:
import zipfile
print(zipfile.is_zipfile('/home/elicer/data_download/181.실내_자율주차용_데이터/01-1.정식개방데이터/Training/01.원천데이터/TS_객체인식(2Hz)_기타(기타)_20220908_10-14_02.camera.zip'))

True


In [2]:
import os
import random
import shutil

train_dir = '/home/elicer/syh/images/train'
val_dir = '/home/elicer/syh/images/val'

# val 디렉토리가 없으면 생성
os.makedirs(val_dir, exist_ok=True)

# train 디렉토리 내 모든 파일 리스트 가져오기
all_files = [f for f in os.listdir(train_dir) if os.path.isfile(os.path.join(train_dir, f))]

# 랜덤 시드 고정
random.seed(42)

# 파일 리스트 섞기
random.shuffle(all_files)

# 9:1 비율로 분할
split_index = int(len(all_files) * 0.9)
train_files = all_files[:split_index]
val_files = all_files[split_index:]

# val 디렉토리로 파일 이동
for file_name in val_files:
    src_path = os.path.join(train_dir, file_name)
    dst_path = os.path.join(val_dir, file_name)
    shutil.move(src_path, dst_path)

print(f"train에 남은 파일 수: {len(train_files)}")
print(f"val로 이동한 파일 수: {len(val_files)}")


train에 남은 파일 수: 1128
val로 이동한 파일 수: 126


In [4]:
import os
import shutil

path_dir1 = '/home/elicer/syh/labels_yolo/train'
path_dir2 = '/home/elicer/syh/images/val'
path_dir3 = '/home/elicer/syh/labels_yolo/val'

# dir2의 파일 이름(확장자 제외) 집합 만들기
base_names_dir2 = set(os.path.splitext(f)[0] for f in os.listdir(path_dir2) if os.path.isfile(os.path.join(path_dir2, f)))

# dir1의 파일 중, 이름(확장자 제외)이 dir2에 있는 것만 이동
for fname in os.listdir(path_dir1):
    src_path = os.path.join(path_dir1, fname)
    if os.path.isfile(src_path):
        base_name = os.path.splitext(fname)[0]
        if base_name in base_names_dir2:
            dst_path = os.path.join(path_dir3, fname)
            shutil.move(src_path, dst_path)


In [3]:
import os

def find_different_files(img_dir, label_dir):
    files_img = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if os.path.isfile(os.path.join(img_dir, f)))
    files_label = set(os.path.splitext(f)[0] for f in os.listdir(label_dir) if os.path.isfile(os.path.join(label_dir, f)))
    
    only_in_img = sorted(list(files_img - files_label))
    only_in_label = sorted(list(files_label - files_img))
    return only_in_img, only_in_label

output_image = "/home/elicer/dataset/images/val"
output_label = "/home/elicer/dataset/labels/val" 

only_in_img, only_in_label = find_different_files(output_image, output_label)
print("이미지에만 있는 파일:", only_in_img)
print("라벨에만 있는 파일:", only_in_label)

이미지에만 있는 파일: []
라벨에만 있는 파일: []


In [20]:
import os

def delete_png_files_with_basenames(directory, basenames):
    deleted_files = []
    for basename in basenames:
        file_path = os.path.join(directory, basename + '.txt')
        if os.path.exists(file_path):
            os.remove(file_path)
            deleted_files.append(file_path)
    return deleted_files

# 사용 예시
directory = '/home/elicer/dataset/labels_segformer/train'  # 실제 png 파일이 있는 폴더로 변경
basenames_to_delete = [
    '20220828_152012_67',
    '20220929_114509_91',
    '20220929_114512_01',
    '20220929_143518_01',
    '20221026_135126_52'
]

deleted_files = delete_png_files_with_basenames(directory, basenames_to_delete)
print("삭제된 파일:", deleted_files)

삭제된 파일: ['/home/elicer/dataset/labels_segformer/train/20220828_152012_67.txt', '/home/elicer/dataset/labels_segformer/train/20220929_114509_91.txt', '/home/elicer/dataset/labels_segformer/train/20220929_114512_01.txt', '/home/elicer/dataset/labels_segformer/train/20220929_143518_01.txt', '/home/elicer/dataset/labels_segformer/train/20221026_135126_52.txt']
